# Labelling Zone

This notebook is responsible of copying data from the exploitation zone (after all the ingestion pipeline and preprocessing has been performed) and generate the labelled data that will be used as the dataset for further fine-tuning of the embedding-generating model (variants of CLIP).

Before executing this notebook, the status is the following:
- There is a bucket named "exploitation-zone" with all processed images and texts
- There is a multimodal collection of embeddings on chromadb that contains baseline CLIP embeddings for all images 

## Preparing new buckets

In [1]:
import boto3
from botocore.exceptions import ClientError
import os
from dotenv import load_dotenv

load_dotenv()
access_key_id = os.getenv("ACCESS_KEY_ID")
secret_access_key = os.getenv("SECRET_ACCESS_KEY")
minio_url = "http://" + os.getenv("S3_API_ENDPOINT")


minio_client = boto3.client(
    "s3",
    aws_access_key_id=access_key_id,
    aws_secret_access_key=secret_access_key,
    endpoint_url=minio_url
)

manifest_name = "dataset_labelled.json"

In [2]:
new_bucket = "labelling-zone"
try:
    minio_client.create_bucket(Bucket=new_bucket)
except ClientError as e:
    error_code = e.response['Error']['Code']
    if error_code in ['BucketAlreadyExists', 'BucketAlreadyOwnedByYou']:
        print(f"Bucket '{new_bucket}' already exists")
    else:
        print(f"Error creating bucket: {e}")

## Copy data

We first will copy the data from one zone to the other so we can keep track of the changes being made to the data.

In [3]:
source_bucket = "exploitation-zone"

response = minio_client.list_objects_v2(Bucket=source_bucket)
if 'Contents' in response:
    for obj in response['Contents']:
        copy_source = {'Bucket': source_bucket, 'Key': obj['Key']}
        minio_client.copy_object(CopySource=copy_source, Bucket=new_bucket, Key=obj['Key'])
        print(f"Copied {obj['Key']} from {source_bucket} to {new_bucket}")
else:
    print(f"No objects found in bucket '{source_bucket}'.")

Copied images/ISIC_0024388.png from exploitation-zone to labelling-zone
Copied images/ISIC_0024508.png from exploitation-zone to labelling-zone
Copied images/ISIC_0024853.png from exploitation-zone to labelling-zone
Copied images/ISIC_0025118.png from exploitation-zone to labelling-zone
Copied images/ISIC_0025200.png from exploitation-zone to labelling-zone
Copied images/ISIC_0025202.png from exploitation-zone to labelling-zone
Copied images/ISIC_0025298.png from exploitation-zone to labelling-zone
Copied images/ISIC_0025343.png from exploitation-zone to labelling-zone
Copied images/ISIC_0025430.png from exploitation-zone to labelling-zone
Copied images/ISIC_0025806.png from exploitation-zone to labelling-zone
Copied images/ISIC_0025874.png from exploitation-zone to labelling-zone
Copied images/ISIC_0025886.png from exploitation-zone to labelling-zone
Copied images/ISIC_0025899.png from exploitation-zone to labelling-zone
Copied images/ISIC_0025960.png from exploitation-zone to labelli

## Automatic labelling
There are many ways to label the data depending on our purposes. One of them is labelling it manually, associating for each image the most similar text and generating a set of pairs that will be fed to the model. 

Nevertheless, in this project we deviate from that approach and focus on fine-tuning the embedding model from the efficiency point-of-view: We aim to achieve the performance expected from a baseline model by fine-tuning a more efficient and less expensive model. We will apply knowledge distillation as defined in the project statement, using the embeddings obtained automatically from the "teacher" model to generate the dataset to fine-tune the "student" model.

For each image we will search for the most similar text and generate pairs automatically. Also, we will keep the similarity score for further selection of the sample.

In [4]:
import chromadb
import json

client = chromadb.HttpClient(host=os.getenv("CHROMADB_ENDPOINT"), port=os.getenv("CHROMADB_PORT"))

collection_text = client.get_collection("text_multimodal_collection")
collection_image = client.get_collection("image_multimodal_collection")
#objects = collection.get(include=["metadatas", "documents"])
image_data = collection_image.get(
    include=["embeddings", "metadatas", "documents"]
)
image_embeddings = image_data["embeddings"]

dataset_pairs = []

results = collection_text.query(
    query_embeddings=image_embeddings,
    n_results=1,
    include=["metadatas", "documents", "distances"],
)
print(results)

for i, img_id in enumerate(image_data["ids"]):
    best_text_id = results["ids"][i][0]
    l2_distance = results["distances"][i][0]
    cosine_similarity = 1 - l2_distance/2
    dataset_pairs.append({
        "image": img_id,
        "text": best_text_id,
        "l2_distance": l2_distance,
        "cosine_similarity": cosine_similarity
    })

with open(manifest_name, "w") as f:
    json.dump(dataset_pairs, f)

minio_client.upload_file(manifest_name, new_bucket, manifest_name)

{'ids': [['texts/basal_cell_carcinoma_0_0_3.txt'], ['texts/melanoma_0_0_155.txt'], ['texts/melanoma_0_0_74.txt'], ['texts/melanoma_0_0_74.txt'], ['texts/melanoma_0_0_68.txt'], ['texts/melanoma_0_0_68.txt'], ['texts/melanoma_0_0_146.txt'], ['texts/melanoma_0_0_164.txt'], ['texts/basal_cell_carcinoma_0_0_47.txt'], ['texts/basal_cell_carcinoma_0_0_47.txt'], ['texts/squamous_cell_carcinoma_0_0_5.txt'], ['texts/basal_cell_carcinoma_0_0_9.txt'], ['texts/melanoma_0_0_24.txt'], ['texts/melanoma_0_0_74.txt'], ['texts/melanoma_0_0_74.txt'], ['texts/melanoma_0_0_68.txt'], ['texts/skin_cancer_0_0_50.txt'], ['texts/basal_cell_carcinoma_0_0_9.txt'], ['texts/melanoma_0_0_24.txt'], ['texts/basal_cell_carcinoma_0_0_13.txt'], ['texts/basal_cell_carcinoma_0_0_3.txt'], ['texts/melanoma_0_0_24.txt'], ['texts/actinic_keratosis_0_0_5.txt'], ['texts/melanoma_0_0_74.txt'], ['texts/melanoma_0_0_74.txt'], ['texts/melanoma_0_0_24.txt'], ['texts/melanoma_0_0_68.txt'], ['texts/basal_cell_carcinoma_0_0_13.txt'], ['t

## Comments
Some observations can be noticed from the resulting dataset:
- It is small, containing only 100 pairs of image-text. 
- The cosine similarity of the pairs generated by the teacher model is not very good (30% on mean), which means that the teacher does not consider them very similar. In other words, two options can be considered:
    - The teacher is not very good and lacks specialization on this area. The model has not been trained enough to be able to confidently know similarities between the pictures of skin cancer and the provided texts
    - If we assume that the teacher model is good, then it must be concluded that the provided dataset is not good, containing images and texts that are very different and contain no relationship between them. 

In [5]:
import pandas as pd
df = pd.read_json(manifest_name)
df.shape
df.head()
df.describe()

,l2_distance,cosine_similarity
count,100.000000,100.000000
mean,1.394165,0.302918
std,0.034663,0.017332
min,1.317803,0.246140
25%,1.374317,0.293748
50%,1.394006,0.302997
75%,1.412504,0.312841
max,1.507721,0.341098
